In [ ]:
# For handling dataset
import pandas as pd

# For numerical operations
import numpy as np

# Splitting data
from sklearn.model_selection import train_test_split

# Encoding & Scaling
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Model
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier


# Evaluation
from sklearn.metrics import accuracy_score, classification_report

# Import Dataset
from google.colab import files

# Upload Files
import io

import joblib

from imblearn.over_sampling import RandomOverSampler

# Upload user_data

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving user_details_new.csv to user_details_new.csv


In [ ]:
df = pd.read_csv(io.BytesIO(uploaded['user_details_new.csv']))

# Display first few rows
print(df.head())

   Age Disability Type  Disability Percentage  Annual Income  \
0   13             all                     79          50000   
1   21        physical                     88         100000   
2    4         hearing                     46         200000   
3   17         hearing                     66         100000   
4   44          visual                     97         700000   

  State Residing In Assistive Devices Required  \
0           Haryana                 Wheelchair   
1         Jharkhand                        NaN   
2               Goa                        NaN   
3            Kerala                 Wheelchair   
4     Uttar Pradesh                        NaN   

                                    Eligible Schemes  
0  Free Education Scheme for Disabled Students, A...  
1                                                NaN  
2  Subsidy Scheme for Assistive Devices for Perso...  
3  Subsidy Scheme for Assistive Devices for Perso...  
4                                      

# Handle Missing Values

In [ ]:
# Replace NaN values with 0 (or use other methods)
df.fillna(0, inplace=True)

***Highly imbalanced data***

In [ ]:
print(df["Eligible Schemes"].value_counts())

Eligible Schemes
0                                                                                                                                                           491
Subsidy Scheme for Assistive Devices for Persons with Disabilities                                                                                          295
Gujarat State Medical Assistance for Disabled, Assistive Technology Support Scheme\n                                                                         17
Free Education Scheme for Disabled Students, Gujarat State Medical Assistance for Disabled                                                                   16
Maharashtra State Disability Welfare Fund, Assistive Technology Support Scheme\n                                                                             16
Special Education And Vocational Training Through Government Institutions, Subsidy Scheme for Assistive Devices for Persons with Disabilities                14
Subsidy Scheme for Assi

In [ ]:
df.columns

Index(['Age', 'Disability Type', 'Disability Percentage', 'Annual Income',
       'State Residing In', 'Assistive Devices Required', 'Eligible Schemes'],
      dtype='object')

In [ ]:
df['Eligible Schemes'] = df['Eligible Schemes'].astype(str).apply(lambda x: x.split(',')[0])

In [ ]:
# Encoding categorical data
state_encoder = LabelEncoder()
df["State Residing In"] = state_encoder.fit_transform(df["State Residing In"])

disability_encoder = LabelEncoder()
df["Disability Type"] = disability_encoder.fit_transform(df["Disability Type"])

# Save encoders and scalers
joblib.dump(state_encoder, "state_encoder.pkl")
joblib.dump(disability_encoder, "disability_encoder.pkl")

['disability_encoder.pkl']

In [ ]:
X = df[['Disability Percentage', 'Annual Income', 'Disability Type', 'Age', 'State Residing In']]
y = df['Eligible Schemes']

oversampler = RandomOverSampler(sampling_strategy='auto', random_state=42)
X_resampled, y_resampled = oversampler.fit_resample(X, y)

df_resampled = pd.DataFrame(X_resampled, columns=X.columns)
df_resampled["Eligible Schemes"] = y_resampled

In [ ]:
df_resampled = df_resampled[df_resampled["Eligible Schemes"].astype(str).str.strip() != "0"]

print("Before Oversampling:")
print(df["Eligible Schemes"].value_counts())

print("\nAfter Oversampling:")
print(df_resampled["Eligible Schemes"].value_counts())


Before Oversampling:
Eligible Schemes
0                                                                                       491
Subsidy Scheme for Assistive Devices for Persons with Disabilities                      315
Gujarat State Medical Assistance for Disabled                                            42
Maharashtra State Disability Welfare Fund                                                36
Free Education Scheme for Disabled Students                                              35
Assistive Technology Support Scheme\n                                                    30
Special Education And Vocational Training Through Government Institutions                27
NIRAMAYA Health Insurance Scheme                                                         10
DISHA (Early Intervention and School Readiness Scheme)                                    5
Assistance to Disabled Persons for Purchase/Fitting of Aids/Appliances (ADIP Scheme)      4
Maharashtra State Handicapped Loan Scheme 

In [ ]:
# Remove "0" before splitting
df_resampled = df_resampled[df_resampled["Eligible Schemes"].astype(str).str.strip() != "0"]

# Now split data
X_train, X_test, y_train, y_test = train_test_split(df_resampled.drop("Eligible Schemes", axis=1),
                                                    df_resampled["Eligible Schemes"],
                                                    test_size=0.2,
                                                    random_state=42)

In [ ]:
age_scaler = MinMaxScaler()
X_train["Age"] = age_scaler.fit_transform(X_train[["Age"]])
X_test["Age"] = age_scaler.transform(X_test[["Age"]])

income_scaler = MinMaxScaler()
X_train["Annual Income"] = income_scaler.fit_transform(X_train[["Annual Income"]])
X_test["Annual Income"] = income_scaler.transform(X_test[["Annual Income"]])

joblib.dump(age_scaler, "age_scaler.pkl")
joblib.dump(income_scaler, "income_scaler.pkl")

['income_scaler.pkl']

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Evaluate model function
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')  # Use 'weighted' for imbalanced data
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(f"Accuracy: {accuracy*100:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Train and evaluate Random Forest
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
print("Random Forest Performance:")
evaluate_model(model, X_test, y_test)

# Train and evaluate Extra Trees
extra_trees = ExtraTreesClassifier(n_estimators=100, max_depth=5, random_state=42)
extra_trees.fit(X_train, y_train)
print("\nExtra Trees Performance:")
evaluate_model(extra_trees, X_test, y_test)

# Train and evaluate Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=5)
gb.fit(X_train, y_train)
print("\nGradient Boosting Performance:")
evaluate_model(gb, X_test, y_test)


Random Forest Performance:
Accuracy: 100.00%
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000

Classification Report:
                                                                                       precision    recall  f1-score   support

Assistance to Disabled Persons for Purchase/Fitting of Aids/Appliances (ADIP Scheme)       1.00      1.00      1.00        92
                                                Assistive Technology Support Scheme
       1.00      1.00      1.00       110
                              DISHA (Early Intervention and School Readiness Scheme)       1.00      1.00      1.00       102
                                         Free Education Scheme for Disabled Students       1.00      1.00      1.00        99
                                              GYAN PRABHA Educational Support Scheme       1.00      1.00      1.00        99
                                       Gujarat State Medical Assistance for Disabled       1.00      1.00      1.00        

In [ ]:
import pandas as pd
from tabulate import tabulate

# Define test cases
test_cases = pd.DataFrame([
    {"Disability Percentage": 80, "Annual Income": 300000, "Disability Type": "mental", "Age": 25, "State Residing In": "Maharashtra"},
    {"Disability Percentage": 60, "Annual Income": 100000, "Disability Type": "physical", "Age": 10, "State Residing In": "Delhi"},
    {"Disability Percentage": 50, "Annual Income": 50000, "Disability Type": "visual", "Age": 18, "State Residing In": "Tamil Nadu"},
    {"Disability Percentage": 70, "Annual Income": 450000, "Disability Type": "hearing", "Age": 45, "State Residing In": "Karnataka"},
    {"Disability Percentage": 85, "Annual Income": 600000, "Disability Type": "mental", "Age": 5, "State Residing In": "Uttar Pradesh"},
])

# Apply encoding to categorical features
test_cases["Disability Type"] = disability_encoder.transform(test_cases["Disability Type"])
test_cases["State Residing In"] = state_encoder.transform(test_cases["State Residing In"])

# Apply normalization to numerical features
test_cases["Age"] = age_scaler.transform(test_cases[["Age"]])
test_cases["Annual Income"] = income_scaler.transform(test_cases[["Annual Income"]])

print("Test Cases:\n")
print(tabulate(test_cases, headers="keys", tablefmt="grid"))

# Predict using each model
rf_preds = model.predict(test_cases)
gb_preds = gb.predict(test_cases)
et_preds = extra_trees.predict(test_cases)

# Print each prediction on a new line
print("Random Forest Predictions:")
for scheme in rf_preds:
    print("-", scheme)

print("\nGradient Boosting Predictions:")
for scheme in gb_preds:
    print("-", scheme)

print("\nExtra Trees Predictions:")
for scheme in et_preds:
    print("-", scheme)


Test Cases:

+----+-------------------------+-----------------+-------------------+-----------+---------------------+
|    |   Disability Percentage |   Annual Income |   Disability Type |       Age |   State Residing In |
+====+=========================+=================+===================+===========+=====================+
|  0 |                      80 |       0.384615  |                 3 | 0.489796  |                  13 |
+----+-------------------------+-----------------+-------------------+-----------+---------------------+
|  1 |                      60 |       0.0769231 |                 4 | 0.183673  |                   4 |
+----+-------------------------+-----------------+-------------------+-----------+---------------------+
|  2 |                      50 |       0         |                 5 | 0.346939  |                  17 |
+----+-------------------------+-----------------+-------------------+-----------+---------------------+
|  3 |                      70 |       0.6

In [ ]:
from sklearn.ensemble import VotingClassifier

# Create the classifiers
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=5)

# Create the voting classifier
voting_clf = VotingClassifier(estimators=[
    ('rf', rf),
    ('gb', gb)
], voting='soft')  # Use 'hard' for majority voting

# Train the ensemble model
voting_clf.fit(X_train, y_train)

# Make predictions
y_pred = voting_clf.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Voting Classifier Accuracy: {accuracy * 100:.2f}%")

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# Base models
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=5)

# Meta-model (can be Logistic Regression, XGBoost, etc.)
meta_model = LogisticRegression()

# Stacking classifier
stacking_clf = StackingClassifier(
    estimators=[('rf', rf), ('gb', gb)],
    final_estimator=meta_model
)

# Train the stacking model
stacking_clf.fit(X_train, y_train)

# Make predictions
y_pred = stacking_clf.predict(X_test)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Stacking Classifier Accuracy: {accuracy * 100:.2f}%")


Voting Classifier Accuracy: 99.83%
Stacking Classifier Accuracy: 99.83%


In [ ]:
import joblib

# Save the trained model
joblib.dump(voting_clf, 'voting_model.pkl')
joblib.dump(stacking_clf, 'stacking_model.pkl')

['stacking_model.pkl']

In [ ]:
encoder_disability = joblib.load("disability_encoder.pkl")
encoder_state = joblib.load("state_encoder.pkl")
scaler_age = joblib.load("age_scaler.pkl")
scaler_income = joblib.load("income_scaler.pkl")
sm = joblib.load("stacking_model.pkl")
vm = joblib.load("voting_model.pkl")

new_input = {
    "Disability Percentage": 80,
    "Annual Income": 300000,
    "Disability Type": "mental",
    "Age": 25,
    "State Residing In": "Maharashtra"
}

new_input["Disability Type"] = disability_encoder.transform([new_input["Disability Type"]])[0]
new_input["State Residing In"] = state_encoder.transform([new_input["State Residing In"]])[0]

new_input_df = pd.DataFrame([new_input], columns=X_train.columns)

# Normalize numerical features using the same MinMaxScaler
new_input_df["Age"] = scaler_age.transform(np.array(new_input_df["Age"]).reshape(-1, 1))[0][0]
new_input_df["Annual Income"] = scaler_income.transform(np.array(new_input_df["Annual Income"]).reshape(-1, 1))[0][0]

# Convert to NumPy array and reshape for model
input_array = np.array([list(new_input.values())]).reshape(1, -1)

# print(f"Original Age: {new_input['Age']}")
# print(f"Normalized Age: {scaler_age.transform(np.array(25).reshape(-1,1))}")

# Make prediction
predicted_scheme_sm = sm.predict(input_array)
predicted_scheme_vm = vm.predict(input_array)

print(f"Predicted Eligible Scheme by SM: {predicted_scheme_sm[0]}")
print(f"Predicted Eligible Scheme by VM: {predicted_scheme_vm[0]}")

Predicted Eligible Scheme: DISHA (Early Intervention and School Readiness Scheme)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(


In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(model, X_train, y_train, cv=5)
print(f"Cross-Validation Accuracy: {np.mean(cv_scores) * 100:.2f}%")


Cross-Validation Accuracy: 99.96%


In [ ]:
print(f"Training Accuracy: {model.score(X_train, y_train) * 100:.2f}%")
print(f"Test Accuracy: {model.score(X_test, y_test) * 100:.2f}%")

Training Accuracy: 100.00%
Test Accuracy: 99.83%


In [ ]:
print(y_resampled.value_counts())

Eligible Schemes
Free Education Scheme for Disabled Students                                             491
0                                                                                       491
Subsidy Scheme for Assistive Devices for Persons with Disabilities                      491
Assistive Technology Support Scheme\n                                                   491
NIRAMAYA Health Insurance Scheme                                                        491
Assistance to Disabled Persons for Purchase/Fitting of Aids/Appliances (ADIP Scheme)    491
Maharashtra State Disability Welfare Fund                                               491
Gujarat State Medical Assistance for Disabled                                           491
Special Education And Vocational Training Through Government Institutions               491
Maharashtra State Handicapped Loan Scheme                                               491
DISHA (Early Intervention and School Readiness Scheme)         

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_resampled, y_resampled, cv=cv, scoring="accuracy")

print(f"Cross-validation Accuracy: {np.mean(cv_scores) * 100:.2f}%")


Cross-validation Accuracy: 99.92%


In [ ]:
X_test.describe()

,Disability Percentage,Annual Income,Disability Type,Age,State Residing In
count,1179.000000,1179.000000,1179.000000,1179.000000,1179.000000
mean,72.253605,0.362498,1.593723,0.325752,10.545377
std,18.663611,0.347927,1.288052,0.292041,5.978337
min,40.000000,0.000000,0.000000,0.000000,0.000000
25%,58.000000,0.076923,1.000000,0.081633,6.000000
50%,75.000000,0.230769,1.000000,0.326531,10.000000
75%,87.000000,0.692308,2.000000,0.489796,16.000000
max,100.000000,1.000000,5.000000,1.000000,19.000000


from matplotlib import pyplot as plt
_df_0['Disability Percentage'].plot(kind='hist', bins=20, title='Disability Percentage')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['Annual Income'].plot(kind='hist', bins=20, title='Annual Income')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2['Disability Type'].plot(kind='hist', bins=20, title='Disability Type')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_3['Age'].plot(kind='hist', bins=20, title='Age')
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
_df_7.plot(kind='scatter', x='Age', y='State Residing In', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_8['Disability Percentage'].plot(kind='line', figsize=(8, 4), title='Disability Percentage')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_9['Annual Income'].plot(kind='line', figsize=(8, 4), title='Annual Income')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_10['Disability Type'].plot(kind='line', figsize=(8, 4), title='Disability Type')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_11['Age'].plot(kind='line', figsize=(8, 4), title='Age')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
X_train.describe()

,Disability Percentage,Annual Income,Disability Type,Age,State Residing In
count,4713.000000,4713.000000,4713.000000,4713.000000,4713.000000
mean,71.701888,0.358599,1.644388,0.333671,10.468067
std,18.456978,0.352575,1.354110,0.295720,5.891230
min,40.000000,0.000000,0.000000,0.000000,0.000000
25%,58.000000,0.076923,1.000000,0.081633,6.000000
50%,73.000000,0.230769,1.000000,0.346939,10.000000
75%,87.000000,0.692308,2.000000,0.551020,16.000000
max,100.000000,1.000000,5.000000,1.000000,19.000000


In [ ]:
import pandas as pd
import numpy as np

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=5)
gb.fit(X_train, y_train)

feature_importances_rf = pd.DataFrame(
    {"Feature": X_train.columns, "Importance": model.feature_importances_}
).sort_values(by="Importance", ascending=False)

feature_importances_gb = pd.DataFrame(
    {"Feature": X_train.columns, "Importance": model.feature_importances_}
).sort_values(by="Importance", ascending=False)

print(feature_importances_rf)
print(feature_importances_gb)

                 Feature  Importance
3                    Age    0.284184
0  Disability Percentage    0.210212
2        Disability Type    0.195013
1          Annual Income    0.168429
4      State Residing In    0.142162
                 Feature  Importance
3                    Age    0.284184
0  Disability Percentage    0.210212
2        Disability Type    0.195013
1          Annual Income    0.168429
4      State Residing In    0.142162


In [ ]:
X_train_new = X_train.drop(columns=["Age"])
X_test_new = X_test.drop(columns=["Age"])

# Retrain the model
model.fit(X_train_new, y_train)
new_accuracy = model.score(X_test_new, y_test)
print(f"New Test Accuracy after removing Age: {new_accuracy:.4f}")

New Test Accuracy after removing Age: 0.9593


In [ ]:
import numpy as np

X_test_noisy = X_test.copy()
X_test_noisy["Age"] += np.random.normal(0, 5, X_test.shape[0])  # Add Gaussian noise

print("Predictions before noise:", model.predict(X_test[:10]))
print("Predictions after noise:", model.predict(X_test_noisy[:10]))

Predictions before noise: ['Subsidy Scheme for Assistive Devices for Persons with Disabilities'
 'Special Education And Vocational Training Through Government Institutions'
 'Free Education Scheme for Disabled Students'
 'Maharashtra State Disability Welfare Fund'
 'NIRAMAYA Health Insurance Scheme' 'NIRAMAYA Health Insurance Scheme'
 'Gujarat State Medical Assistance for Disabled'
 'Subsidy Scheme for Assistive Devices for Persons with Disabilities'
 'Gujarat State Medical Assistance for Disabled'
 'Gujarat State Medical Assistance for Disabled']
Predictions after noise: ['Subsidy Scheme for Assistive Devices for Persons with Disabilities'
 'Special Education And Vocational Training Through Government Institutions'
 'GYAN PRABHA Educational Support Scheme'
 'Maharashtra State Disability Welfare Fund'
 'NIRAMAYA Health Insurance Scheme'
 'Free Education Scheme for Disabled Students'
 'Gujarat State Medical Assistance for Disabled'
 'Subsidy Scheme for Assistive Devices for Persons with

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42)

X_train_new = X_train.drop(columns=["Annual Income"])
X_test_new = X_test.drop(columns=["Annual Income"])

rf.fit(X_train_new, y_train)
gb.fit(X_train_new, y_train)

print(f"New Accuracy GB: {gb.score(X_test_new, y_test):.4f}")
print(f"New Accuracy RF: {rf.score(X_test_new, y_test):.4f}")

New Accuracy GB: 0.9924
New Accuracy RF: 0.9771


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 4, 5],
    "subsample": [0.7, 0.8, 0.9]
}

grid_search = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.9}


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# Base models
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=5)

# Meta-model
meta_model = LogisticRegression()

# Stacking classifier
stacking_clf = StackingClassifier(
    estimators=[('rf', rf), ('gb', gb)],
    final_estimator=meta_model
)

# Train with the latest data
stacking_clf.fit(X_train_new, y_train)

# Save the retrained model
joblib.dump(stacking_clf, 'stacking_model_updated.pkl')


['stacking_model_updated.pkl']

In [ ]:
df_resampled["Age_Disability"] = df_resampled["Age"].astype(str) + "_" + df_resampled["Disability Type"].astype(str)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, random_state=42)
rf.fit(X_train, y_train)

test_cases = pd.DataFrame([
    {"Disability Percentage": 60, "Annual Income": 100000, "Disability Type": "physical", "Age": 10, "State Residing In": "Delhi"},
    {"Disability Percentage": 80, "Annual Income": 900000, "Disability Type": "visual", "Age": 3, "State Residing In": "Gujarat"},
])

test_cases["Disability Type"] = disability_encoder.transform(test_cases["Disability Type"])
test_cases["State Residing In"] = state_encoder.transform(test_cases["State Residing In"])

test_cases["Age"] = age_scaler.transform(test_cases[["Age"]])
test_cases["Annual Income"] = income_scaler.transform(test_cases[["Annual Income"]])

predictions = rf.predict(test_cases)
print(predictions)


['Subsidy Scheme for Assistive Devices for Persons with Disabilities'
 'Subsidy Scheme for Assistive Devices for Persons with Disabilities'
 'Special Education And Vocational Training Through Government Institutions']


In [ ]:
test_cases = pd.DataFrame([
    {"Disability Percentage": 60, "Annual Income": 100000, "Disability Type": "physical", "Age": 10, "State Residing In": "Delhi"},
    {"Disability Percentage": 80, "Annual Income": 900000, "Disability Type": "visual", "Age": 3, "State Residing In": "Gujarat"},
])

test_cases["Disability Type"] = disability_encoder.transform(test_cases["Disability Type"])
test_cases["State Residing In"] = state_encoder.transform(test_cases["State Residing In"])

test_cases["Age"] = age_scaler.transform(test_cases[["Age"]])
test_cases["Annual Income"] = income_scaler.transform(test_cases[["Annual Income"]])

stacking_model = joblib.load("stacking_model_updated.pkl")
predictions = stacking_model.predict(test_cases)
print(predictions)

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Annual Income


In [ ]:
income_scaler = joblib.load("income_scaler.pkl")
age_scaler = joblib.load("age_scaler.pkl")

# Sample values to normalize
age_value = np.array([[4]])  # Single value for Age
income_value = np.array([[17]])  # Single value for Income (replace with actual income value)

# Transform values (without fitting again)
normalized_age = age_scaler.transform(age_value)
normalized_income = income_scaler.transform(income_value)

print("Normalized Age:", normalized_age)
print("Normalized Income:", normalized_income)

Normalized Age: [[0.06122449]]
Normalized Income: [[-0.07689692]]


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
